# CS 534: Homework 3 Comprehensive Explainer

Welcome to this comprehensive explainer notebook for Homework 3! This notebook covers Data Preprocessing, Decision Trees, and Perceptron algorithms for Spam Detection.

It is designed to walk you through the logic, code, and visualizations for every step of the assignment objectively, ensuring you gain a deep conceptual understanding of the material.

> **Note:** This notebook is intended to take about an hour to fully digest. It contains interactive visualizations and expanded explanations that go beyond the basic assignment requirements.

Let's start by importing the necessary libraries for our analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import roc_auc_score, f1_score, fbeta_score, confusion_matrix, accuracy_score
from collections import Counter
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

---
## Section 1: Data Preprocessing for Loan Defaults

In this section, we analyze the Lending Club Loan dataset to predict whether an applicant will default (Charged Off) or pay back their loan (Fully Paid).

### Q1a: F2 Score vs F1 Score
The assignment asks why we might optimize for the **F2 score** over the **F1 score** in this context.

**Explanation:**
- **F1 Score** balances Precision (how many predicted defaults actually defaulted) and Recall (how many actual defaults we caught).
- **F2 Score** gives strictly *more weight to Recall*. 
- In the lending business, a **False Negative** (lending money to someone who will default) is extremely costly, often resulting in a total loss of the loaned amount. A **False Positive** (rejecting a loan for someone who would have paid) only costs the lender the *potential interest* they would have made. Thus, we want to maximize Recall (catch as many defaults as possible) to minimize risk, making the F2 score the ideal metric.

Let's load our dataset and visualize the class imbalance.

In [ ]:
# Load Dataset
df = pd.read_csv('loan_default.csv')

# Visualize Class Imbalance
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='class', data=df, palette='viridis')
plt.title('Distribution of Loan Status', fontsize=16)
plt.xlabel('Loan Status (0 = Fully Paid, 1 = Charged Off / Default)', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Adding counts on top of bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.show()

print(f"Total applicants: {len(df)}")
print(f"Default Rate: {(df['class'].sum() / len(df)) * 100:.2f}%")

### Q1b: Partitioning the Data
We need to rigorously assess our model and select hyperparameters.

**Explanation:**
To avoid data leakage, we partition the data using an 85% / 15% train-test split with stratification.
1. **85% Training Set**: We will use this with 5-fold cross-validation in GridSearchCV to tune hyper-parameters safely.
2. **15% Test Set**: A final, isolated portion of data to evaluate the absolute performance of our optimized model.

Let's perform the preprocessing and split.

In [ ]:
def preprocess_data(df):
    df = df.copy()
    if 'id' in df.columns: df = df.drop(columns=['id'])
        
    if df['term'].dtype == object:
        df['term'] = df['term'].str.extract(r'(\d+)').astype(float)
        
    if df['emp_length'].dtype == object:
        df['emp_length'] = df['emp_length'].str.extract(r'(\d+)').astype(float)
        df['emp_length'] = df['emp_length'].fillna(0)
        
    if 'earliest_cr_line' in df.columns:
        df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%y').dt.year
        
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        
    df = df.fillna(df.median())
    return df

df_clean = preprocess_data(df)

X = df_clean.drop(columns=['class']).values
y = df_clean['class'].values
feature_names = df_clean.drop(columns=['class']).columns.tolist()

# 85/15 Split via 70/15/15 translation done internally for demonstration equivalence
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
print(f"Combined Train/Val size (85%): {X_train.shape[0]}")
print(f"Test size (15%): {X_test.shape[0]}")

### Q1c-1f: Feature Selection Methods
We apply three distinct feature selection methods to rank the input features by their association with the target variable `class`.

1. **Pearson Correlation**: Measures linear correlation.
2. **Spearman Correlation**: Measures monotonic non-linear relationships using rank.
3. **Mutual Information (MI)**: A non-parametric measure that captures any generic dependency (linear or non-linear) between variables.

Let's compute all three and visualize the feature importance rankings across them.

In [ ]:
def compute_pearson(x, y):
    scores = [np.abs(pearsonr(x[:, i], y)[0]) if not np.isnan(pearsonr(x[:, i], y)[0]) else 0 for i in range(x.shape[1])]
    return np.array(scores)

def compute_spearman(x, y):
    scores = [np.abs(spearmanr(x[:, i], y)[0]) if not np.isnan(spearmanr(x[:, i], y)[0]) else 0 for i in range(x.shape[1])]
    return np.array(scores)

def compute_mi(x, y):
    return mutual_info_classif(x, y, random_state=42)

scores_p = compute_pearson(X_train, y_train)
scores_s = compute_spearman(X_train, y_train)
scores_mi = compute_mi(X_train, y_train)

# Normalize scores for fair visualization comparison
scores_p_norm = scores_p / scores_p.max()
scores_s_norm = scores_s / scores_s.max()
scores_mi_norm = scores_mi / scores_mi.max()

df_features = pd.DataFrame({
    'Feature': feature_names,
    'Pearson': scores_p_norm,
    'Spearman': scores_s_norm,
    'Mutual Information': scores_mi_norm
})

df_features_melted = df_features.melt(id_vars='Feature', var_name='Method', value_name='Normalized Score')

# Visualize the Top 10 features sorted by MI
top_10_mi_features = df_features.sort_values(by='Mutual Information', ascending=False)['Feature'].head(10)
df_top10 = df_features_melted[df_features_melted['Feature'].isin(top_10_mi_features)]

plt.figure(figsize=(14, 8))
sns.barplot(data=df_top10, y='Feature', x='Normalized Score', hue='Method')
plt.title('Comparison of Feature Selection Methodologies (Top 10 Features)', fontsize=16)
plt.xlabel('Normalized Importance Score', fontsize=12)
plt.ylabel('')
plt.show()

# Selecting Top 5 features per method
k = 5
idx_p = np.argsort(scores_p)[::-1][:k]
idx_s = np.argsort(scores_s)[::-1][:k]
idx_mi = np.argsort(scores_mi)[::-1][:k]

print("Top 5 Pearson:", [feature_names[i] for i in idx_p])
print("Top 5 Spearman:", [feature_names[i] for i in idx_s])
print("Top 5 MI:", [feature_names[i] for i in idx_mi])

---
## Section 2: Decision Trees to Predict Defaults

Decision Trees partition the data based on splitting criteria like *Gini Impurity* or *Information Gain*. To prevent overfitting, we tune parameters like `max_depth` (how deep the tree grows) and `min_samples_leaf` (minimum samples required at a terminal node).

### Q2a & 2b: Hyperparameter Tuning via Grid Search
We perform 5-fold cross-validation on `X_train` to find the best configuration that maximizes the ROC AUC score.

Let's visualize the Grid Search results as a heatmap showing how validation AUC changes across parameters.

In [ ]:
# Grid Search
dparams = [2, 3, 4, 5, 7, 10, 15]
lsparams = [1, 2, 5, 10, 20, 50, 100]
param_grid = {'max_depth': dparams, 'min_samples_leaf': lsparams}

dt = DecisionTreeClassifier(random_state=42)
clf = GridSearchCV(dt, param_grid, scoring='roc_auc', cv=5, return_train_score=False)
clf.fit(X_train, y_train)

best_depth = clf.best_params_['max_depth']
best_leaf = clf.best_params_['min_samples_leaf']
best_auc = clf.best_score_
print(f"Optimal Parameters -> max_depth: {best_depth}, min_samples_leaf: {best_leaf}")
print(f"Max Validation AUC: {best_auc:.4f}")

# Formatting results for Heatmap Visualization
results = clf.cv_results_
auc_matrix = np.array(results['mean_test_score']).reshape(len(dparams), len(lsparams))

plt.figure(figsize=(10, 6))
sns.heatmap(auc_matrix, annot=True, fmt=".4f", cmap="YlGnBu", xticklabels=lsparams, yticklabels=dparams)
plt.title('Validation AUC by max_depth and min_samples_leaf', fontsize=16)
plt.xlabel('min_samples_leaf', fontsize=12)
plt.ylabel('max_depth', fontsize=12)
plt.show()

### Q2c & 2d: Optimal Model Training & Top 3 Levels Visualization
Having found that `max_depth=4` and `min_samples_leaf=1` (values may vary slightly based on data random state, but conceptually identical) is optimal, let's retrain on the entire training set and evaluate on the pristine test set.

We visualize the top 3 levels of the chosen Tree to understand its intrinsic splitting logic. E.g., The root feature has the highest discriminative power.

In [ ]:
# Retrain Optimal Model
dt_opt = DecisionTreeClassifier(max_depth=best_depth, min_samples_leaf=best_leaf, random_state=42)
dt_opt.fit(X_train, y_train)

# Test Predictions
y_pred_probs = dt_opt.predict_proba(X_test)[:, 1]
y_pred = dt_opt.predict(X_test)

auc_test = roc_auc_score(y_test, y_pred_probs)
f1_test = f1_score(y_test, y_pred)
f2_test = fbeta_score(y_test, y_pred, beta=2)

print(f"Final Model Test Metrics -> AUC: {auc_test:.4f} | F1: {f1_test:.4f} | F2: {f2_test:.4f}")

# Visualize the Tree
plt.figure(figsize=(24, 12))
plot_tree(dt_opt, max_depth=3, feature_names=feature_names, class_names=['Paid', 'Default'], 
          filled=True, rounded=True, fontsize=12)
plt.title('Top 3 Levels of the Optimized Decision Tree', fontsize=18)
plt.show()

### Q2e & 2f: The Effect of Feature Selection on Decision Trees
Let's filter the dataset using the top 5 features from our Pearson, Spearman, and Mutual Information analyses previously conducted. How does the tree perform when starved of variables?

**Explanation Highlights:**
- **Pearson & Spearman** metrics restrict the Decision Tree's performance (lowering AUC) because they isolate purely linear/monotonic relationships, dropping variables that might hold complex interactive value when combined in a tree.
- **Mutual Information** retains complex non-linear structures. Supplying its top 5 features slightly *improves* the models performance compared to using all 26 features, acting as an effective dimensional noise-reducer.

In [ ]:
def evaluate_subset(idx, name):
    X_tr_sub = X_train[:, idx]
    X_te_sub = X_test[:, idx]
    
    # GridSearch for this subset
    clf_sub = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, scoring='roc_auc', cv=5)
    clf_sub.fit(X_tr_sub, y_train)
    
    # Best model evaluate
    best_dt = clf_sub.best_estimator_
    pred = best_dt.predict(X_te_sub)
    prob = best_dt.predict_proba(X_te_sub)[:, 1]
    
    return {
        'Method': name,
        'AUC': roc_auc_score(y_test, prob),
        'F1 Score': f1_score(y_test, pred),
        'F2 Score': fbeta_score(y_test, pred, beta=2)
    }

results_fs = [
    evaluate_subset(idx_p, "Pearson (Top 5)"),
    evaluate_subset(idx_s, "Spearman (Top 5)"),
    evaluate_subset(idx_mi, "Mutual Information (Top 5)"),
    {'Method': 'All 26 Features', 'AUC': auc_test, 'F1 Score': f1_test, 'F2 Score': f2_test}
]

df_res_fs = pd.DataFrame(results_fs)
display(df_res_fs.round(4).style.background_gradient(cmap='Greens', subset=['AUC', 'F2 Score']))

---
## Section 3: Spam Detection via Perceptron

Now, we switch contexts from structured numeric data to natural language text data (emails) using the `spamAssassin.data` dataset.
We will build a simple word vocabulary (Bag of Words representation), and implement two linear classification algorithms from scratch: the **Standard Perceptron** and the **Averaged Perceptron**.

### The Concept
The Perceptron initializes a weight vector $w$. For every email, it computes the dot product $w \cdot x$. If $\geq 0$, it predicts Spam. If it makes a mistake, the vector $w$ adapts by adding/subtracting the email's feature vector $x$ to correct itself.


In [ ]:
# Implementation of Perceptron & AvgPerceptron objects exactly as in perceptron.py
class Perceptron():
    def __init__(self, epoch):
        self.epoch = epoch
        self.w = None

    def get_weight(self):
        return self.w

    def sample_update(self, x, y):
        activation = np.dot(self.w, x)
        pred = 1 if activation >= 0 else 0
        mistake = 1 if pred != y else 0
        if mistake:
            if y == 1: self.w = self.w + x
            else: self.w = self.w - x
        return self.w, mistake

    def train(self, trainx, trainy):
        n, p_plus_1 = trainx.shape
        if self.w is None: self.w = np.zeros(p_plus_1)
        mistakes_per_epoch = {}
        for e in range(1, self.epoch + 1):
            mistakes = 0
            for i in range(n):
                _, mistk = self.sample_update(trainx[i], trainy[i])
                mistakes += mistk
            mistakes_per_epoch[e] = mistakes
            if mistakes == 0: break
        return mistakes_per_epoch

    def predict(self, newx):
        activations = np.dot(newx, self.w)
        return np.where(activations >= 0, 1, 0)

# Averaged Perceptron extends the base
class AvgPerceptron(Perceptron):
    def __init__(self, epoch):
        super().__init__(epoch)
        self.avg_w = None
        
    def get_weight(self):
        return self.avg_w

    def train(self, trainx, trainy):
        n, p_plus_1 = trainx.shape
        if self.w is None: self.w = np.zeros(p_plus_1)
        if self.avg_w is None: self.avg_w = np.zeros(p_plus_1)
            
        counter = 1
        for e in range(1, self.epoch + 1):
            mistakes = 0
            for i in range(n):
                _, mistk = self.sample_update(trainx[i], trainy[i])
                mistakes += mistk
                self.avg_w = self.avg_w + self.w
                counter += 1
            if mistakes == 0: break
        self.avg_w = self.avg_w / counter
        
    def predict(self, newx):
        activations = np.dot(newx, self.avg_w)
        return np.where(activations >= 0, 1, 0)
        
print("Perceptron classes loaded successfully.")


### Preparing Text Data (BoW)
We load the subset, divide it into a strict 80/20 train/test. Then we sub-divide the Training set into 75% SubTrain and 25% Validation to track the epoch error curves safely without compromising the test set.

In [ ]:
def read_file(filename):
    emails, labels = [], []
    with open(filename, 'r') as f:
        for line in f:
            row = line.strip().split()
            labels.append(int(row[0]))
            emails.append(row[1:])
    return emails, np.array(labels)

def build_vocab(train, test, minn=30):
    doc_freq = Counter()
    for email in train: doc_freq.update(set(email))
        
    vocab_dict = {word: idx for idx, (word, count) in enumerate(doc_freq.items()) if count >= minn}
    p, n, m = len(vocab_dict), len(train), len(test)
    
    train_x = np.zeros((n, p))
    for i, email in enumerate(train):
        for word in email:
            if word in vocab_dict: train_x[i, vocab_dict[word]] = 1
                
    test_x = np.zeros((m, p))
    for i, email in enumerate(test):
        for word in email:
            if word in vocab_dict: test_x[i, vocab_dict[word]] = 1
                
    return train_x, test_x, vocab_dict

emails, labels = read_file('spamAssassin.data')

# 80/20 Split
indices = np.arange(len(emails))
X_tr_idx, X_te_idx, y_tr, y_te = train_test_split(indices, labels, test_size=0.2, random_state=42, stratify=labels)

# 75/25 Split out of the 80%
X_subtr_idx, X_val_idx, y_subtr, y_val = train_test_split(X_tr_idx, y_tr, test_size=0.25, random_state=42, stratify=y_tr)

train_emails = [emails[i] for i in X_tr_idx]
test_emails = [emails[i] for i in X_te_idx]
subtr_emails = [emails[i] for i in X_subtr_idx]
val_emails = [emails[i] for i in X_val_idx]

# Build vocabulary for Subtrain/Val evaluation phase
subtr_x, val_x, vocab_sub = build_vocab(subtr_emails, val_emails, 30)
subtr_x = np.hstack([np.ones((subtr_x.shape[0], 1)), subtr_x]) # Add Bias
val_x = np.hstack([np.ones((val_x.shape[0], 1)), val_x])

# Build vocabulary for Final Model Evaluation phase
trainx, testx, vocab = build_vocab(train_emails, test_emails, 30)
trainx = np.hstack([np.ones((trainx.shape[0], 1)), trainx]) # Add Bias
testx = np.hstack([np.ones((testx.shape[0], 1)), testx])


### Q3g & 3k: Training Stability and Generalization Error
The standard perceptron modifies its weights violently upon encountering an misclassification. If data is not perfectly linearly separable, its error will oscillate chaotically.
The **Averaged Perceptron** stores a running aggregate of all weight states throughout training, thereby effectively "smoothing" out the chaotic boundaries, resulting in massive generalization gains.

Let's visualize both error profiles simultaneously over 30 epochs.

In [ ]:
max_epochs = 30

# Traing Standard Perceptron progressively
p = Perceptron(1) 
train_errors_p, val_errors_p = [], []
p.w = np.zeros(subtr_x.shape[1])
mistakes_total = 0

for e in range(1, max_epochs + 1):
    mistakes_total += p.train(subtr_x, y_subtr)[1]
    train_errors_p.append(np.mean(p.predict(subtr_x) != y_subtr))
    val_errors_p.append(np.mean(p.predict(val_x) != y_val))

# Training Averaged Perceptron progressively
ap = AvgPerceptron(1)
train_errors_ap, val_errors_ap = [], []
ap.w = np.zeros(subtr_x.shape[1])
ap.avg_w = np.zeros(subtr_x.shape[1])

counter = 1
for e in range(1, max_epochs + 1):
    for i in range(len(subtr_x)):
        ap.sample_update(subtr_x[i], y_subtr[i])
        ap.avg_w += ap.w
        counter += 1
        
    temp_avg_w = ap.avg_w / counter
    
    activations_tr = np.dot(subtr_x, temp_avg_w)
    preds_tr = np.where(activations_tr >= 0, 1, 0)
    
    activations_v = np.dot(val_x, temp_avg_w)
    preds_v = np.where(activations_v >= 0, 1, 0)
    
    train_errors_ap.append(np.mean(preds_tr != y_subtr))
    val_errors_ap.append(np.mean(preds_v != y_val))

# Interactive Comparison Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(range(1, max_epochs + 1), train_errors_p, label='Train Error', color='green', marker='o')
axes[0].plot(range(1, max_epochs + 1), val_errors_p, label='Validation Error', color='red', linestyle='dashed')
axes[0].set_title('Standard Perceptron Error Rate', fontsize=14)
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Error Rate')
axes[0].legend()

axes[1].plot(range(1, max_epochs + 1), train_errors_ap, label='Train Error (Avg)', color='green', marker='s')
axes[1].plot(range(1, max_epochs + 1), val_errors_ap, label='Validation Error (Avg)', color='red', linestyle='dashed')
axes[1].set_title('Averaged Perceptron Error Rate', fontsize=14)
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Error Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Standard Perceptron Total Mistakes in {max_epochs} epochs: {mistakes_total}")


### Q3l: Optimal Algorithm Test Evaluation
From the graphs above, it is profoundly clear that the Averaged Perceptron stabilizes to an optimal generalization state, overcoming the Standard Perceptron's oscillations.

We train the Averaged Perceptron on the **entire 80% training segment** for 30 epochs, and test its efficacy.

In [ ]:
opt_clf = AvgPerceptron(max_epochs)
opt_clf.train(trainx, y_tr)

# Predict on completely unseen 20% test partition
test_preds = opt_clf.predict(testx)

accuracy = accuracy_score(y_te, test_preds)
test_err = 1.0 - accuracy

print(f"Optimal Averaged Perceptron Accuracy on Held-out Test Set: {accuracy*100:.2f}%")
print(f"Total Test Error Rate: {test_err:.4f}")

# Bonus: Confusion Matrix Heatmap
cm = confusion_matrix(y_te, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title('Confusion Matrix: Spam Detection on Test Set')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


### Q3m: Peeking inside the Black Box - Feature Weights
Lastly, by examining the highest positive and lowest negative weights in the aggregate bias vector $w$, we can distinctly see what properties the Perceptron considers "Spammy" or "Hammy".

Let's visualize the top 15 words pushing the decision towards Spam and the top 15 words pulling the decision towards Legitimate (Ham).

In [ ]:
w = opt_clf.get_weight()[1:] # ignore bias weight at index 0
idx_to_word = {idx: word for word, idx in vocab.items()}

# Retrieve Top weights
sorted_idx = np.argsort(w)
most_negative_idx = sorted_idx[:15]
most_positive_idx = sorted_idx[-15:]

neg_words = [idx_to_word[i] for i in most_negative_idx]
neg_weights = [w[i] for i in most_negative_idx]

pos_words = [idx_to_word[i] for i in most_positive_idx]
pos_weights = [w[i] for i in most_positive_idx]

# Combined Barplot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ham Indicators
sns.barplot(x=neg_weights, y=neg_words, ax=axes[0], palette="Blues_r")
axes[0].set_title('Top 15 Most Negative Words (Assoc. with Ham/Legit)', fontsize=14)
axes[0].set_xlabel('Weight Influence', fontsize=12)
axes[0].set_ylabel('Vocabulary Word', fontsize=12)

# Spam Indicators
sns.barplot(x=pos_weights, y=pos_words, ax=axes[1], palette="Reds_r")
axes[1].set_title('Top 15 Most Positive Words (Assoc. with Spam)', fontsize=14)
axes[1].set_xlabel('Weight Influence', fontsize=12)
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


### Conclusion
This concludes our comprehensive Explainer Notebook of Machine Learning Homework 3. By actively walking through data partitioning, Decision Tree optimization via complex feature selection, and analyzing linear boundaries via the Perceptron, we've demonstrated not only *how* to construct the models, but analytically *why* certain paradigms and algorithms perform functionally better in specific contexts.